# チュートリアル 08: 高度なマルチエージェントワークフロー

##  学習目標
このノートブックを完了すると、以下ができるようになります:
- 複雑なルーティングのために `WorkflowBuilder` で **カスタムワークフローグラフ** を構築する
- ビジネスロジックを持つ **CustomExecutor** を作成する
- 並列処理のための **ファンアウト/ファンインパターン** を実装する
- `MagenticBuilder` で **AI を利用したオーケストレーション** を使用する
- **条件付きルーティング** と複雑なワークフローパターンを処理する
- 基本ワークフローと比較して **高度なパターンをいつ使用するか** を理解する

##  前提条件

このチュートリアルを始める前に、以下を完了しておく必要があります:
- **チュートリアル 05: マルチエージェントワークフロー** (SequentialBuilder、ConcurrentBuilder)

このチュートリアルは、これらの概念をより高度なパターンで構築します!

##  何が「高度」なのか?

**基本ワークフロー (チュートリアル 05):**
-  シンプルで宣言的なパターン
-  順次または並行実行
-  カスタムロジック不要
- 例: `SequentialBuilder`、`ConcurrentBuilder`

**高度なワークフロー (このチュートリアル):**
-  CustomExecutor にカスタムビジネスロジック
-  条件付きルーティングと分岐
-  カスタムゲートを持つファンアウト/ファンイン
-  AI を利用した動的オーケストレーション
- 例: `WorkflowBuilder`、`MagenticBuilder`

### 高度なパターンをいつ使用するか

| ユースケース | パターン | 理由 |
|----------|---------|------|
| 承認ワークフロー | WorkflowBuilder | 承認ステータスに基づく条件付きルーティングが必要 |
| 予算検証ゲート | WorkflowBuilder | 続行前に検証が必要 |
| 多段階パイプライン | WorkflowBuilder | カスタム制御フローが必要 |
| 動的タスク計画 | MagenticBuilder | 最適なワークフローが事前にわからない |
| 適応型システム | MagenticBuilder | AI に調整戦略を決定させる必要がある |

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
from typing import cast

from agent_framework import (
    # 高度なワークフロービルダー
    WorkflowBuilder,
    # ワークフローコンポーネント
    Executor,
    AgentExecutor,
    handler,
    WorkflowContext,
    # 追跡のためのイベント
    WorkflowEvent,
    AgentResponseUpdate,
    # 基本型
    Message,
)
from agent_framework.orchestrations import MagenticBuilder
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)
print(" インポート成功!")
print("📦 高度なワークフロービルダー準備完了: WorkflowBuilder、MagenticBuilder")

In [ ]:
import os
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# 環境変数ベースで設定（Agent Framework は標準の OTEL_* を読む）
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Metrics は無効化（必要に応じて変更）

# enable_sensitive_data=True を指定して機微データ(OpenAI の IN/OUT データ)の収集を有効化
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    
    Args:
        workflow: 可視化するワークフローオブジェクト
        filename: 保存するファイル名（拡張子なし）
        show_preview: プレビューを表示するかどうか
    
    Returns:
        保存されたSVGファイルのパス
    """
    # WorkflowVizオブジェクトを作成
    viz = WorkflowViz(workflow)
    
    # 3. SVGファイルとして保存
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVGファイルを保存しました: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. SVGをプレビュー表示
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ エラー: 'graphviz'パッケージがインストールされていません")
        print("インストール方法: pip install agent-framework[viz] --pre")
        print(f"詳細: {e}")
        return None
    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        return None

## ステップ 2: 専門エージェントを作成

チュートリアル 05 の旅行エージェント専門家を再利用します。

In [ ]:
async def create_travel_agents():
    """
    専門旅行計画エージェントを作成します。
    """
    # 注意: セッション終了の問題を避けるためコンテキストマネージャなしで AzureCliCredential を使用
    # 認証情報は複数のエージェント呼び出しで再利用されます
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    # Azure OpenAI クライアントの作成
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    # フライト専門家
    flight_agent = chat_client.as_agent(
        instructions="""
        あなたはフライト予約の専門家です。
        以下を考慮した簡潔で実用的なフライトの推奨事項を提供してください：
        - 予約に最適なタイミング
        - 航空会社の好みと品質
        - 接続戦略
        - 価格と利便性のトレードオフ
        
        回答は簡潔に（最大2〜3文）。
        """,
        name="FlightExpert",
    )
    
    # ホテル専門家
    hotel_agent = chat_client.as_agent(
        instructions="""
        あなたはホテル予約の専門家です。
        以下を考慮した簡潔なホテルの推奨事項を提供してください：
        - 観光客に最適な地域
        - コストパフォーマンス
        - 観光名所や交通機関への近さ
        - ホテルの品質と設備
        
        回答は簡潔に（最大2〜3文）。
        """,
        name="HotelExpert",
    )
    
    # アクティビティ専門家
    activities_agent = chat_client.as_agent(
        instructions="""
        あなたは地元のアクティビティと体験の専門家です。
        以下を考慮した簡潔なアクティビティの推奨事項を提供してください：
        - 必見の観光名所
        - 地元の人気スポットと隠れた名所
        - 文化的な体験
        - 食事とグルメ
        
        回答は簡潔に（最大2〜3文）。
        """,
        name="ActivitiesExpert",
    )
    
    return flight_agent, hotel_agent, activities_agent

print(" エージェントファクトリー作成完了")

## ステップ 3: CustomExecutor - WorkflowBuilder

**CustomExecutor** を使用すると、ワークフローにビジネスロジックを追加できます:
- 検証ゲート (続行前に条件をチェック)
- データ変換 (エージェント間でメッセージを変更)
- 条件付きルーティング (ロジックに基づいて異なるエージェントに送信)
- 集約ロジック (カスタム方法で結果を結合)

### CustomExecutor の作成

すべての CustomExecutor は以下を行う必要があります:
1. **`Executor` を継承する**
2. **`__init__` で `super().__init__(id="unique_id")` を呼ぶ**
3. **`WorkflowContext` 型注釈を持つ `@handler` メソッドを定義する**
4. **`ctx.send_message()` を使用して** 下流の実行者にデータを渡す

In [ ]:
# 予算検証用の CustomExecutor
class BudgetValidator(Executor):
    """計画エージェントに進む前に旅行予算を検証します。"""
    
    def __init__(self):
        # 必須: 一意の ID で super().__init__() を呼ぶ
        super().__init__(id="budget_validator")
    
    @handler
    async def validate_budget(self, request: str, ctx: WorkflowContext[str]):
        """
        予算が言及されているかチェックし、下流のエージェントに転送します。
        
        これは以下を実演します:
        - ビジネスロジックの実行 (予算キーワード検出)
        - メッセージ変換 (コンテキストの追加)
        - ctx.send_message() を介した下流の実行者への渡し
        """
        print(f"    予算検証: リクエストを分析中...")
        
        # ビジネスロジック: 予算関連キーワードをチェック
        budget_keywords = ['budget', '$', 'price', 'cost', 'cheap', 'expensive', 'affordable']
        has_budget = any(word in request.lower() for word in budget_keywords)
        
        if has_budget:
            print(f"    予算の考慮事項が検出されました")
            # 予算コンテキストでメッセージを変換
            enhanced_request = f"予算重視の旅行リクエスト: {request}"
        else:
            print(f"     予算が指定されていません - 標準価格を想定")
            # 予算のリマインダーを追加
            enhanced_request = f"標準旅行リクエスト (中価格帯のオプションを検討): {request}"
        
        # 下流のエージェントに送信 (ファンアウトはエッジを介して発生)
        await ctx.send_message(enhanced_request)

print(" BudgetValidator 実行者作成完了")

## ステップ 4: カスタムワークフローグラフの構築

**WorkflowBuilder** はワークフローグラフの完全な制御を提供します:
- 実行者とエッジを手動で追加
- ファンアウトパターンを作成 (1 → 多数)
- ファンインパターンを作成 (多数 → 1)
- 条件付きルーティングを追加
- 複雑な DAG を構築

### グラフ構築パターン

```python
workflow = (
    WorkflowBuilder()
    .set_start_executor(start_node)           # エントリポイントを定義
    .add_fan_out_edges(validator, [a, b, c])  # 1 → 多数
    .add_edge(a, aggregator)                   # 個別のエッジ
    .add_edge(b, aggregator)
    .add_edge(c, aggregator)
    .build()                                   # 検証して作成
)
```

In [ ]:

"""
WorkflowBuilder を実演: 検証ゲートとファンアウト/ファンインを持つカスタム DAG。

ワークフローグラフ:
ユーザー入力 → 予算検証 → 3エージェントにファンアウト → 集約 → 最終出力
"""
print("="*70)
print("カスタムワークフロー: 予算検証 + 並列エージェント + 集約")
print("="*70)
print("\nワークフローグラフ:")
print("  ユーザー → 予算検証 → ファンアウト ┬→ フライト専門家")
print("                                    ├→ ホテル専門家    → 集約")
print("                                    └→ アクティビティ専門家")
print()

# 必要な型をインポート
from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
)
from typing import Never

# すべてのエージェントレスポンスを収集してフォーマットする集約を作成
class TravelPlanAggregator(Executor):
    """すべての旅行エージェントからレスポンスを収集し、最終プランをフォーマットします"""
    
    def __init__(self):
        super().__init__(id="travel_plan_aggregator")
    
    @handler
    async def aggregate(
        self, 
        results: list[AgentExecutorResponse], 
        ctx: WorkflowContext[Never, str]
    ) -> None:
        """すべてのエージェントレスポンスをフォーマットされた旅行プランに集約します"""
        # エージェントごとにレスポンスを抽出 (サイレントに - ここで print なし)
        responses_by_agent = {}
        for result in results:
            agent_id = result.executor_id
            response_text = result.agent_response.text if result.agent_response else ""
            if response_text:
                responses_by_agent[agent_id] = response_text
        
        # 統合プランをフォーマット
        plan = "\n" + "="*70 + "\n"
        plan += "📋 あなたの完全な予算バルセロナ旅行プラン\n"
        plan += "="*70 + "\n\n"
        
        if "FlightExpert" in responses_by_agent:
            plan += "  フライト:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['FlightExpert']}\n\n"
        
        if "HotelExpert" in responses_by_agent:
            plan += " 宿泊施設:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['HotelExpert']}\n\n"
        
        if "ActivitiesExpert" in responses_by_agent:
            plan += "🎭 アクティビティと体験:\n"
            plan += "─"*70 + "\n"
            plan += f"{responses_by_agent['ActivitiesExpert']}\n\n"
        
        plan += "="*70 + "\n"
        plan += " すべての推奨事項はリクエスト通り予算重視です!\n"
        plan += "="*70
        
        await ctx.yield_output(plan)

# エージェントを作成
flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# エージェントを AgentExecutor でラップ
flight_executor = AgentExecutor(flight_agent, id="FlightExpert")
hotel_executor = AgentExecutor(hotel_agent, id="HotelExpert")
activities_executor = AgentExecutor(activities_agent, id="ActivitiesExpert")

# カスタム実行者を作成
budget_validator = BudgetValidator()
aggregator = TravelPlanAggregator()

# ワークフローを構築
workflow = (
    WorkflowBuilder(start_executor=budget_validator)
    .add_fan_out_edges(
        budget_validator,
        [flight_executor, hotel_executor, activities_executor]
    )
    .add_fan_in_edges(
        [flight_executor, hotel_executor, activities_executor],
        aggregator
    )
    .build()
)

# 関数を使ってワークフローを可視化
svg_path = visualize_workflow(
    workflow, 
    filename="advanced_multi_agent_workflow.svg",
    show_preview=True
)


In [ ]:
from contextlib import nullcontext

request = "バルセロナへの予算重視の旅行を計画してください。安いフライト、手頃なホテル、無料のアクティビティが必要です。"
print(f" ユーザーリクエスト:")
print(f"{'─'*70}")
print(f"  {request}")
print(f"{'─'*70}\n")

print(" ワークフロー実行中...\n")
print(" 実行フロー:")
print("="*70)

# クリーンな出力で実行を追跡 (重複なし)
validator_shown = False
step2_shown = False
aggregator_shown = False
agents_completed = set()
final_plan = None

# ★ start_as_current_span() でワークフロー内部スパンを子としてネスト
cm = tracer.start_as_current_span("AdvancedWorkflowExecution") if tracer else nullcontext()
with cm:
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent):
            if event.type == "executor_invoked":
                if event.executor_id == "budget_validator" and not validator_shown:
                    print(" ステップ 1: 予算検証")
                    print("   → 予算キーワードのリクエストを分析中...")
                    validator_shown = True
                elif event.executor_id in ["FlightExpert", "HotelExpert", "ActivitiesExpert"]:
                    if not step2_shown:
                        print("\n ステップ 2: 専門エージェントにファンアウト (並列)")
                        step2_shown = True
                elif event.executor_id == "travel_plan_aggregator" and not aggregator_shown:
                    print("\n ステップ 3: 結果を集約中")
                    print("   → すべてのエージェントからレスポンスを収集中...")
                    aggregator_shown = True
            
            elif event.type == "executor_completed":
                if event.executor_id not in agents_completed:
                    agents_completed.add(event.executor_id)
                    print(f"   ✓ {event.executor_id}")
            
            elif event.type == "output":
                if isinstance(event.data, list):
                    final_plan = event.data
                elif isinstance(event.data, str):
                    final_plan = event.data
                elif not isinstance(event.data, AgentResponseUpdate):
                    final_plan = event.data

    # 最終プランを表示 (一度だけ)
    if final_plan:
        print(f"\n{final_plan}\n")
    else:
        print("\n 最終プランが生成されませんでした\n")

    print("="*70)
    print(" ワークフロー完了!")
    print("="*70)
    print("\n 何が起こったか:")
    print("  1, 予算検証が予算キーワードを検出 ✓")
    print("  2, 予算コンテキストでリクエストを強化 ✓")
    print("  3, 3エージェントに並列でファンアウト ✓")
    print("  4, 各エージェントが専門的なアドバイスを提供 ✓")
    print("  5, 集約がすべてのレスポンスを収集してフォーマット ✓")
    print("\n WorkflowBuilder の主な利点:")
    print("   カスタムビジネスロジック (予算検証)")
    print("   並列実行 (ファンアウトパターン)")
    print("   結果の集約 (ファンインパターン)")
    print("   ワークフロー構造の完全な制御")
    print("   決定論的でデバッグ可能な実行")


## 検証ポイント

このパターン（`WorkflowBuilder` + `BudgetValidator` + fan-out/fan-in + `TravelPlanAggregator`）は、**LLMの推論能力ではなく、決定論的ワークフロー制御基盤の正しさ**を検証しています。

---

### 1. CustomExecutor による「前処理ゲート」の動作検証

10_advanced_workflows.ipynb は `Executor` を継承し `@handler` を定義するだけで、**LLMを介さないビジネスロジックノード**をワークフローグラフに差し込めることを証明しています。

フレームワーク内部では、[`@handler` デコレータ](jp/use/agent-framework-main-1.0.0b260130/python/packages/core/agent_framework/_workflows/_executor.py) がメソッドシグネチャから入力型・出力型を自動抽出し、`_handler_spec` としてメタデータを付与します。つまり `BudgetValidator` の `validate_budget(self, request: str, ctx: WorkflowContext[str])` という型注釈だけで、フレームワークが「`str` を受け取り `str` を送出するノード」と認識し、グラフの型整合性を保証します。

**検証ポイント**: `ctx.send_message()` でメッセージを**変換・強化**してから下流に流せるか。

---

### 2. `add_fan_out_edges` による構造化並列分岐

WorkflowBuilder.add_fan_out_edges (L533–L612) は内部で `FanOutEdgeGroup` を生成し、source のメッセージを**全 target へ並列ブロードキャスト**します。

```python
.add_fan_out_edges(budget_validator, [flight_executor, hotel_executor, activities_executor])
```

これは `ConcurrentBuilder` のような高レベルAPIと異なり、**任意のカスタムExecutorを起点にできる**点が本質です。公式サンプル fan_out_fan_in_edges.py も `DispatchToExperts(Executor)` → 3エージェントへの fan-out という同一構造で、これが基本パターンであることを示しています。

---

### 3. `add_fan_in_edges` による型安全な集約

add_fan_in_edges (L830–L920) は全 source の完了を待ち、結果を **`list[T]` として集約**して target に渡します。

```python
.add_fan_in_edges([flight_executor, hotel_executor, activities_executor], aggregator)
```

`TravelPlanAggregator` の `@handler` が `results: list[AgentExecutorResponse]` を受け取る設計は、フレームワークの**型制約**（fan-in target の handler 引数は `list[...]` でなければならない）に準拠しています。そして `ctx.yield_output()` を使って**ワークフロー最終出力**を送出する点も、`ctx.send_message()`（次のExecutorへ転送）との責務分離を検証しています。

---

### 4. イベント駆動の可観測性

実行セル（[cell 14](jp/use/10_advanced_workflows.ipynb#L356)）で `ExecutorInvokedEvent`、`AgentRunEvent`、`WorkflowOutputEvent` を `run_stream()` から取得しています。これは単なるログではなく、**ワークフローグラフの実行がイベントストリームとして外部から観測・制御可能であること**の検証です。OpenTelemetry トレーサーとの連携（テストファイル `test_workflow_observability.py` にも対応テストあり）により、本番環境でのデバッグ・監視が可能な設計であることを示しています。

---

### 5. `ConcurrentBuilder` との本質的な違い

フレームワーク内部の `ConcurrentBuilder` は同じ fan-out/fan-in を内蔵の `_DispatchToAllParticipants` と `_AggregateAgentConversations` で行う**高レベルAPI**です。このノートブックがあえて `WorkflowBuilder` を使うのは：

| 観点 | ConcurrentBuilder | WorkflowBuilder (このパターン) |
|------|-------------------|-------------------------------|
| 前処理ロジック | 不可 | `BudgetValidator` で任意のゲート挿入可 |
| 集約ロジック | 固定（全応答結合） | `TravelPlanAggregator` で自由にフォーマット |
| グラフ構造 | 暗黙的 | 明示的DAG定義 + `WorkflowViz` で可視化 |
| 拡張性 | Agent限定 | 任意のExecutor（非LLMノード含む） |

---

### まとめ

このパターンが本質的に問うているのは：

> **「LLMに判断を委ねず、開発者がグラフ構造として実行フローを定義し、カスタムビジネスロジック（検証・変換・集約）を型安全に組み込み、決定論的かつ可観測な並列処理パイプラインを構築できるか」**


## ステップ 5: AI を利用したオーケストレーション - MagenticBuilder

**MagenticBuilder** は、AI オーケストレーターを使用してワークフローを動的に作成および管理します:

### 仕組み

1. **あなたが提供**: 利用可能なエージェントのセット
2. **AI オーケストレーター**: 
   - ユーザーのリクエストを分析
   - 実行プランを作成
   - 呼び出すエージェントを選択
   - 実行順序を決定
   - 結果に基づいてプランを適応

### MagenticBuilder をいつ使用するか

 **使用する場合:**
- タスク要件が複雑で予測不可能
- 最適なワークフローが事前にわからない
- 中間結果に基づく適応的計画が必要
- タスクがオープンエンドの問題解決を含む

 **使用しない場合:**
- ワークフローが明確で予測可能
- 保証された実行順序が必要
- パフォーマンスが重要 (AI 計画はオーバーヘッドを追加)
- シンプルな順次または並列パターンで十分

### アーキテクチャ

```
ユーザーリクエスト
     ↓
AI オーケストレーター (LLM)
     ↓
動的プランを作成
     ↓
必要に応じてエージェントを実行
     ↓
結果に基づいてプランを適応
```

In [ ]:
"""
MagenticBuilder を実演: AI を利用した動的オーケストレーション。

AI オーケストレーターは:
1. リクエストを分析
2. 実行プランを作成
3. 呼び出すエージェントと順序を決定
4. 中間結果に基づいて適応
"""
print("="*70)
print("MAGENTIC ワークフロー: AI を利用した動的オーケストレーション")
print("="*70)
print("\n🧠 AI オーケストレーションの仕組み:")
print("  1. AI オーケストレーターがリクエストを分析")
print("  2. 動的実行プランを作成")
print("  3. エージェントをインテリジェントに選択して調整")
print("  4. 中間結果に基づいてプランを適応")
print("  5. いつ、どの順序でエージェントを呼ぶかを決定\n")

# エージェントを作成
flight_agent, hotel_agent, activities_agent = await create_travel_agents()

chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

# マネージャーエージェントを作成
manager_agent = chat_client.as_agent(
    instructions="""
    あなたは複数の旅行専門家を調整するタスクマネージャーです。
    ユーザーのリクエストを分析し、適切な専門家を選択して委任し、
    最終的な旅行プランを統合してください。
    """,
    name="TravelCoordinator",
)

# magentic ワークフローを構築 - AI オーケストレーターがすべてを調整
workflow = (
    MagenticBuilder(
        participants=[flight_agent, hotel_agent, activities_agent],
        manager_agent=manager_agent,
        max_round_count=10,
        max_stall_count=3,
        max_reset_count=2,
    )
    .build()
)


# 関数を使ってワークフローを可視化
svg_path = visualize_workflow(
    workflow, 
    filename="magentic_multi_agent_workflow.svg",
    show_preview=True
)


In [ ]:

# オーケストレーターがどのエージェントをどの順序で呼ぶかを決定
request = (
    "ヴェネツィアへの4日間のロマンチックな記念日旅行を計画する必要があります。"
    "サンマルコ近くの素敵なホテル、ロマンチックなレストラン、ゴンドラ乗りが欲しいです。"
    "何をすべきですか?"
)

print(f" ユーザーリクエスト:")
print(f"{'─'*70}")
print(f"  {request}")
print(f"{'─'*70}\n")

print(" AI オーケストレーション開始中...\n")
print(" 実行フロー:")
print("="*70)

# 何が起こるかを追跡
executor_invoked = []
agent_runs = []
final_output = None

# ★ start_as_current_span() でワークフロー内部スパンを子としてネスト
cm = tracer.start_as_current_span("MagenticWorkflowExecution") if tracer else nullcontext()
with cm:
    # ワークフローを実行してイベントを追跡
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent):
            if event.type == "executor_invoked":
                executor_invoked.append(event.executor_id)
                # オーケストレーターのアクティビティを表示
                if "orchestrator" in event.executor_id.lower():
                    print("   🧠 AI オーケストレーター: 実行プランを作成中...")
                else:
                    agent_name = event.executor_id.replace('agent_', '').replace('_expert', ' 専門家')
                    print(f"   🤖 委任先: {agent_name}")
            
            elif event.type == "executor_completed":
                if event.executor_id and event.executor_id not in agent_runs:
                    agent_runs.append(event.executor_id)
                    agent_name = event.executor_id.replace('agent_', '').replace('_expert', ' 専門家')
                    print(f"   ✓ {agent_name} 完了")
            
            elif event.type == "output":
                if isinstance(event.data, list):
                    final_output = event.data
                elif not isinstance(event.data, AgentResponseUpdate):
                    final_output = event.data

    # 何が起こったかを表示
    print("\n" + "="*70)
    print(" オーケストレーションサマリー")
    print("="*70 + "\n")

    print(f"🧠 オーケストレーション詳細:")
    print(f"   • 呼び出された実行者: {len(executor_invoked)}")
    print(f"   • 委任されたエージェント: {len([e for e in executor_invoked if 'expert' in e.lower()])}")
    print(f"   • オーケストレーター計画ラウンド: {len([e for e in executor_invoked if 'orchestrator' in e.lower()])}")

    agents_called = [e for e in executor_invoked if 'expert' in e.lower()]
    if agents_called:
        print(f"\n🤖 呼び出されたエージェント (順序):")
        for agent_id in agents_called:
            clean_name = agent_id.replace('agent_', '').replace('_expert', ' 専門家')
            print(f"   • {clean_name}")

    # 最終結果を表示
    if final_output:
        print("\n" + "="*70)
        print(" 最終オーケストレーション結果")
        print("="*70 + "\n")
        
        # Message からテキストを抽出
        result_text = None
        if hasattr(final_output, 'text'):
            result_text = final_output.text
        elif isinstance(final_output, str):
            result_text = final_output
        elif isinstance(final_output, list):
            # メッセージのリストかもしれない
            texts = []
            for item in final_output:
                if hasattr(item, 'text'):
                    texts.append(item.text)
                elif isinstance(item, str):
                    texts.append(item)
            result_text = " ".join(texts) if texts else None
        
        if result_text:
            # テキストを整形してラップ
            words = result_text.split()
            line = ""
            for word in words:
                if len(line + word) > 66:
                    print(f"  {line}")
                    line = word + " "
                else:
                    line += word + " "
            if line:
                print(f"  {line.strip()}")
        else:
            print(f"  型: {type(final_output)}")
            print(f"  データ: {final_output}")
        print()
    else:
        print("\n 最終出力がキャプチャされませんでした\n")

    print("="*70)
    print(" MAGENTIC ワークフロー完了!")
    print("="*70)
    print("\n 何が起こったか:")
    print(f"  1, AI オーケストレーターがロマンチックなヴェネツィアリクエストを分析")
    print(f"  2, 動的に {len(agents_called)} エージェントを呼び出すように選択")
    print(f"  3, {len([e for e in executor_invoked if 'orchestrator' in e.lower()])} 計画ラウンドを作成")
    print(f"  4, 最終旅行プランを統合")
    print(f"  5, 調整ロジックを一切コーディングする必要がありませんでした!")
    print("\n MagenticBuilder の主な利点:")
    print("   AI が最適な実行プランを動的に作成")
    print("   リクエストの複雑さに自動的に適応")
    print("   ハードコードされたワークフローロジック不要")
    print("   簡単にスケール - 専門エージェントを追加するだけ")
    print("   予測不可能で複雑なリクエストを優雅に処理")


## 「オーケストレーション自体をLLMに委ねる」ことの成立性検証

1番目のパターンが「開発者がDAGを明示定義する決定論的制御」を問うたのに対し、MagenticBuilder は **「どのエージェントを、どの順序で、何回呼ぶかをLLMが動的に判断する」** という非決定論的オーケストレーションが実用レベルで機能するかを検証しています。

---

### 1. 内部アーキテクチャ: 星型グラフへの自動変換

MagenticBuilder.build() の内部では、**実は WorkflowBuilder を使っています**:

```python
def build(self) -> Workflow:
    participants = self._resolve_participants()
    orchestrator = self._resolve_orchestrator(participants)
    
    workflow_builder = WorkflowBuilder().set_start_executor(orchestrator)
    for participant in participants:
        workflow_builder = workflow_builder.add_edge(orchestrator, participant)
        workflow_builder = workflow_builder.add_edge(participant, orchestrator)
    return workflow_builder.build()
```

つまり内部グラフは **オーケストレーター ↔ 全エージェント間の双方向エッジ**（星型トポロジー）です:

```
         ┌──── FlightExpert
         │
Orchestrator ──── HotelExpert
         │
         └──── ActivitiesExpert
         (全て双方向)
```

1番目のパターンの一方向DAG（`BudgetValidator → fan-out → fan-in → Aggregator`）とは根本的に異なり、**どのパスを通るかはグラフ構造ではなくLLMの判断で決まります**。

---

### 2. Task Ledger: 実行前の事実整理と計画立案

オーケストレーションループに入る**前に**、マネージャーLLMが **`_MagenticTaskLedger`** を生成します。これは `plan()` メソッド（`_magentic.py` L610–L637）で行われる2段階のLLM呼び出しです:

**フェーズ1: Facts（事実の整理）** — `ORCHESTRATOR_TASK_LEDGER_FACTS_PROMPT` をLLMに送り、タスクに関する事実を4カテゴリに分類:

```
1. GIVEN OR VERIFIED FACTS   — リクエストに明示されている事実
2. FACTS TO LOOK UP          — 調べる必要がある事実と情報源
3. FACTS TO DERIVE           — 推論・計算で導出すべき事実
4. EDUCATED GUESSES           — 記憶・推測に基づく事実
```

**フェーズ2: Plan（計画の策定）** — `ORCHESTRATOR_TASK_LEDGER_PLAN_PROMPT` で、チーム構成と事実を踏まえた箇条書きの実行計画を生成。

この2つが `_MagenticTaskLedger(facts=facts_msg, plan=plan_msg)` として保存され、`ORCHESTRATOR_TASK_LEDGER_FULL_PROMPT` で統合されてオーケストレーターの文脈に組み込まれます。

**リセット時の更新**: ストール検出後の `replan()` では `ORCHESTRATOR_TASK_LEDGER_FACTS_UPDATE_PROMPT`（「何が新たにわかったか」を反映）と `ORCHESTRATOR_TASK_LEDGER_PLAN_UPDATE_PROMPT`（「前回何が失敗したか」を踏まえた新計画）でTask Ledgerを再生成します。

**検証ポイント**: 「いきなりエージェントを呼ぶ」のではなく、**事実の構造化 → 計画立案 → 計画に基づくラウンド実行 → 失敗時の再計画** という認知的プロセスをLLMに強制できるか。

---

### 3. Progress Ledger: LLMによるラウンド毎の制御判断

Task Ledger が「実行前の計画」であるのに対し、Progress Ledger は**各ラウンドの制御判断**です。`MagenticOrchestrator` のループ内で `create_progress_ledger()` が呼ばれ、マネージャーLLMが **`MagenticProgressLedger`** をJSON構造体として出力します:

```python
# Progress Ledger のフィールド:
{
    "is_request_satisfied": bool,    # タスク完了か
    "is_in_loop": bool,              # ループに陥っているか
    "is_progress_being_made": bool,  # 前進しているか
    "next_speaker": str,             # 次に呼ぶエージェント名
    "instruction_or_question": str   # そのエージェントへの指示
}
```

**2つのLedgerの関係**: Task Ledger の facts/plan が `chat_history` に含まれた状態で Progress Ledger が生成されるため、Progress Ledger は**Task Ledger の計画を参照しながら「次の一手」を判断**します。

**検証ポイント**: LLMが毎ラウンド「誰を次に呼ぶか」を構造化判断し、それがワークフローの実行フローを動的に決定できるか。1番目の `add_fan_out_edges` のような静的ルーティングとは対照的に、**ルーティング自体がLLM推論の出力**です。

---

### 4. 安全弁: ストール検出とリセット機構

ノートブックで設定されている3つのパラメータは、**LLMの非決定性に対するガードレール**です:

| パラメータ | ノートブックの値 | 制御対象 |
|---|---|---|
| `max_round_count=10` | オーケストレーション総ラウンド数の上限。超過で強制終了 |
| `max_stall_count=3` | 連続して前進がないラウンドの許容数。超過でリセット＆再計画 |
| `max_reset_count=2` | リセット（再計画）の最大回数。超過で強制終了 |

内部フローは:

```
progress_being_made=False → stall_count++
stall_count > max_stall_count → 全参加者リセット + 再計画
reset_count >= max_reset_count → ワークフロー強制終了
round_index >= max_round_count → ワークフロー強制終了
```

`MagenticAgentExecutor` は `handle_magentic_reset()` ハンドラを持ち、リセット時にキャッシュ・会話履歴・スレッドをすべてクリアします。これは1番目のパターンには存在しない、**LLMオーケストレーション特有の耐障害設計**です。

---

### 5. 実行セルのイベント追跡が検証していること

10_advanced_workflows.ipynb で `ExecutorInvokedEvent` / `AgentRunEvent` を追跡し:

- `orchestrator` の呼び出し回数 → **LLMが何回「考えた」か**
- `expert` の呼び出し順序 → **LLMがどの順序でエージェントを選んだか**

を可視化しています。1番目のパターンではこれらは**グラフ定義から予測可能**でしたが、ここでは**実行するまでわからない**。同じリクエストでも実行ごとに異なるエージェント呼び出し順序になりうることそのものが検証対象です。

---

### 6. 2つのパターンの本質的対比

| 観点 | パターン1 (WorkflowBuilder) | パターン2 (MagenticBuilder) |
|------|---------------------------|---------------------------|
| **「誰を呼ぶか」の決定** | 開発者がグラフで定義 | LLMが毎ラウンド判断 |
| **実行順序** | 決定論的・再現可能 | 非決定論的・適応的 |
| **前処理/後処理** | CustomExecutor で自由に実装 | オーケストレーターが自動統合 |
| **並列実行** | fan-out で明示的に並列化 | LLMが逐次的に1エージェントずつ呼ぶ |
| **エラーリカバリ** | 開発者が実装 | ストール検出→リセット→再計画が組み込み |
| **内部実装** | 開発者定義のDAG | `build()` が星型グラフを自動生成 |
| **追加エージェント** | グラフ再定義が必要 | `.participants()` に追加するだけ |
| **ランタイムオーバーヘッド** | 低い | 高い (AI 計画) |
| **最適な用途** | 既知のワークフロー | 未知/複雑なタスク |
| **デバッグ** | 容易 (明示的グラフ) | 困難 (AI の決定) |
| **スケーラビリティ** | 手動更新が必要 | 新しいエージェントに自動適応 |

---

### まとめ

> **「ワークフローの制御フロー自体をLLMに委ね、Task Ledger による事前の事実整理・計画立案 + Progress Ledger による毎ラウンドの構造化判断 + ストール検出/リセットのガードレールを組み合わせることで、事前にフローを定義できない複雑なタスクに対して実用的なマルチエージェント協調が成立するか」**

1番目のパターンが「AIの非決定性をワークフロー制御の決定性で囲い込む」設計だったのに対し、2番目は逆に **「ワークフロー制御自体にAIの非決定性を活用しつつ、安全弁（max_round/stall/reset）で暴走を防ぐ」** 設計です。この2つを対比的に配置することで、全体として**制御の所在を開発者 or AI のどちらに置くか**という設計判断のトレードオフを示しています。

##  高度なパターンの比較

### WorkflowBuilder と MagenticBuilder

### 実世界のユースケース

**WorkflowBuilder の例:**
```python
# 承認ワークフロー
workflow = (
    WorkflowBuilder()
    .set_start_executor(request_handler)
    .add_edge(request_handler, manager_approval)
    .add_switch_case_edge_group(
        manager_approval,
        [
            Case(lambda x: x.approved, finance_review),
            Default(rejection_handler)
        ]
    )
    .build()
)

# 多段階パイプライン
workflow = (
    WorkflowBuilder()
    .add_chain([data_validator, processor, quality_check, publisher])
    .build()
)

# ファンアウト/ファンインパターン
workflow = (
    WorkflowBuilder()
    .set_start_executor(splitter)
    .add_fan_out_edges(splitter, [worker1, worker2, worker3])
    .add_fan_in_edges([worker1, worker2, worker3], aggregator)
    .build()
)
```

**MagenticBuilder の例:**
```python
# リサーチアシスタント
workflow = (
    MagenticBuilder()
    .participants([web_researcher, academic_searcher, data_analyst, summarizer])
    .with_manager(
        agent=manager_agent,
        max_round_count=10
    )
    .build()
)

# カスタマーサポート
workflow = (
    MagenticBuilder()
    .participants([product_expert, billing_expert, tech_support, escalation_handler])
    .with_manager(
        agent=manager_agent,
        max_round_count=8
    )
    .build()
)
```

### 主な違い

**WorkflowBuilder:**
-  実行グラフ全体を定義
-  予測可能、テスト可能、再現可能
-  コンプライアンス、監査に適している
-  事前設計が必要
-  柔軟性のない構造

**MagenticBuilder:**
-  AI オーケストレーターが実行プランを作成
-  複雑で多様なリクエストに適応
-  新しい専門エージェントを簡単に追加
-  予測可能性が低い
-  AI の決定をデバッグするのが困難

##  重要なポイント

### 学んだこと

1. **CustomExecutor**
   - `Executor` を継承し、`super().__init__(id="...")` を呼ぶ
   - `WorkflowContext` 型注釈で `@handler` を使用
   - エージェントに渡す前にビジネスロジックを実装
   - 例: 検証、変換、ルーティング

2. **WorkflowBuilder**
   - ワークフローグラフ構造の完全な制御
   - ファンアウト、ファンイン、条件付きルーティングをサポート
   - 明確に定義された複雑なワークフローに最適
   - 決定論的でデバッグ可能

3. **MagenticBuilder**
   - AI オーケストレーターが実行を動的に管理
   - 予測不可能で適応的なワークフローに最適
   - 多くの専門エージェントに簡単にスケール
   - オーバーヘッドは高いが柔軟性が高い

4. **適切なパターンの選択**
   - **既知のワークフロー** → WorkflowBuilder
   - **未知/適応的** → MagenticBuilder
   - **シンプルな順次** → SequentialBuilder (チュートリアル 05)
   - **シンプルな並列** → ConcurrentBuilder (チュートリアル 05)

### 本番パターン

```python
# CustomExecutor テンプレート
class MyExecutor(Executor):
    def __init__(self):
        super().__init__(id="my_executor")
    
    @handler
    async def process(self, input: str, ctx: WorkflowContext[str]):
        # ここにビジネスロジック
        result = do_something(input)
        await ctx.send_message(result)

# 検証ゲート付き WorkflowBuilder
workflow = (
    WorkflowBuilder()
    .set_start_executor(validator)
    .add_fan_out_edges(validator, [agent1, agent2, agent3])
    .build()
)

# 適応的タスク用 MagenticBuilder
workflow = (
    MagenticBuilder()
    .participants([specialist1, specialist2, specialist3])
    .with_manager(agent=manager_agent, max_round_count=10)
    .build()
)
```

##  演習問題

### 演習 1: 多段階承認ワークフロー

条件付きルーティングを持つワークフローを作成:
```
リクエスト → 検証 → (有効な場合) → マネージャー承認
                                  ├─→ (承認された場合) → 財務
                                  └─→ (拒否された場合) → 拒否ハンドラー
```

**ヒント:** 条件付きルーティングには `add_switch_case_edge_group()` を使用。

### 演習 2: データ処理パイプライン

ファンアウト/ファンインパターンを作成:
```
データスプリッター → ワーカー 1 ↘
                 → ワーカー 2  → 集約 → 最終プロセッサ
                 → ワーカー 3 ↗
```

**ヒント:** `add_fan_out_edges()` と `add_fan_in_edges()` を使用。

### 演習 3: 適応型リサーチシステム

以下を持つ magentic ワークフローを作成:
- Web 研究者
- 学術検索  
- データアナリスト
- ファクトチェッカー
- 要約者

AI オーケストレーターにそれらをどのように調整するか決定させましょう!

In [ ]:
# 演習プレイグラウンド - ここにソリューションを実装してください!

async def approval_workflow_exercise():
    """
    演習 1: 多段階承認ワークフローを構築。
    """
    # TODO: 条件付きルーティングで承認ワークフローを実装
    pass

async def pipeline_exercise():
    """
    演習 2: ファンアウト/ファンインデータ処理パイプラインを構築。
    """
    # TODO: データ処理パイプラインを実装
    pass

async def research_system_exercise():
    """
    演習 3: 適応型リサーチシステムを構築。
    """
    # TODO: magentic リサーチワークフローを実装
    pass

print(" 演習テンプレート準備完了 - 上記のソリューションを実装してください!")

##  次のステップ

おめでとうございます! 高度なマルチエージェントワークフローパターンをマスターしました!

これで以下ができるようになりました:
-  ビジネスロジックを持つ CustomExecutor を構築
-  WorkflowBuilder で複雑なワークフローグラフを作成
-  ファンアウト/ファンインパターンを実装
-  MagenticBuilder で AI を利用したオーケストレーションを使用
-  ユースケースに適したパターンを選択


---

### クイックリファレンス

**CustomExecutor:**
```python
class MyExecutor(Executor):
    def __init__(self):
        super().__init__(id="my_id")
    
    @handler
    async def process(self, msg: str, ctx: WorkflowContext[str]):
        result = transform(msg)
        await ctx.send_message(result)
```

**WorkflowBuilder:**
```python
workflow = (
    WorkflowBuilder()
    .set_start_executor(start)
    .add_fan_out_edges(start, [a, b, c])
    .add_edge(a, end)
    .build()
)
```

**MagenticBuilder:**
```python
workflow = (
    MagenticBuilder()
    .participants([agent1, agent2, agent3])
    .with_manager(agent=manager_agent, max_round_count=10)
    .build()
)
```

**ワークフローを実行:**
```python
# ストリーミング
async for event in workflow.run_stream(input):
    if isinstance(event, AgentRunEvent):
        print(event.data)

# 非ストリーミング
result = await workflow.run(input)
outputs = result.get_outputs()
```